In [12]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent
sys.path.append(str(ROOT))

from src.config import LEAGUES
from src.utils.helper_io import read_json

SR_BRONZE = ROOT / "data" / "bronze" / "sportradar"

In [ ]:
def has_stat(match: dict, key: str) -> bool:
    comps = match.get("statistics", {}).get("totals", {}).get("competitors", [])
    return len(comps) == 2 and all(key in c.get("statistics", {}) for c in comps)

rows = []
for league in LEAGUES:
    for file in sorted((SR_BRONZE / league).glob("season_summary_*.json")):
        for m in read_json(file)["summaries"]:
            event = m["sport_event"]
            ctx = event["sport_event_context"]
            rows.append({
                "league": league,
                "season": ctx["season"]["year"],
                "match_id": event["id"],
                "start_time": event["start_time"],
                "status": m["sport_event_status"]["status"],
                "group": (ctx.get("groups") or [{}])[0].get("name"),
                "stage_type": ctx["stage"]["type"],
                "has_possession": has_stat(m, "ball_possession"),
                "has_shots": has_stat(m, "shots_total"),
                "has_sot": has_stat(m, "shots_on_target"),
            })


matches = pd.DataFrame(rows)
print(matches.shape, "| duplicate match_ids:", matches["match_id"].duplicated().sum())

In [8]:
completed_matches = matches[matches["status"] == "closed"]
uncompleted_matches = matches[matches["status"] != "closed"]
print("Completed Matches:", len(completed_matches))
print("Uncompleted Matches:", len(uncompleted_matches))

Completed Matches: 2508
Uncompleted Matches: 43


In [ ]:
coverage = (
    completed_matches.groupby(["league", "season"])
    .agg(closed=("match_id", "count"),
         possession=("has_possession", "mean"),
         shots=("has_shots", "mean"),
         shots_on_target=("has_sot", "mean"))
    .round(3)
)
coverage.insert(0, "all_rows", matches.groupby(["league", "season"]).size())
coverage

In [10]:
matches["group_clean"] = matches["group"].str.replace(r" \d{2}/\d{2}| \d{4}", "", regex=True)

(matches.groupby(["league", "season", "group_clean", "status"]).size()
        .unstack("status", fill_value=0))

status                                                              closed  \
league         season group_clean                                            
belgian_pro    24/25  Pro League                                       240   
                      Pro League, Championship Round                    30   
                      Pro League, Europa Playoffs                        1   
                      Pro League, Qualifying Round                      30   
                      Pro League, Relegation Round                      12   
                      Pro League, Relegation/Promotion Playoffs          8   
               25/26  Pro League                                       240   
                      Pro League, Championship Round                    30   
                      Pro League, Europa Playoffs                        1   
                      Pro League, Qualifying Round                      30   
                      Pro League, Relegation Round                      12   
                      Pro League, Relegation/Promotion Playoffs          8   
eliteserien    2024   Eliteserien                                      240   
                      Eliteserien, Relegation/Promotion                  5   
               2025   Eliteserien                                      240   
                      Eliteserien, Relegation/Promotion                  5   
liga_portugal  24/25  Liga Portugal                                    306   
                      Liga Portugal, Relegation/Promotion Playoffs       2   
               25/26  Liga Portugal                                    306   
                      Liga Portugal, Relegation/Promotion Playoffs       2   
premier_league 24/25  Premier League                                   380   
               25/26  Premier League                                   380   

status                                                              postponed  
league         season group_clean                                              
belgian_pro    24/25  Pro League                                            2  
                      Pro League, Championship Round                        0  
                      Pro League, Europa Playoffs                           0  
                      Pro League, Qualifying Round                          0  
                      Pro League, Relegation Round                          0  
                      Pro League, Relegation/Promotion Playoffs             0  
               25/26  Pro League                                            3  
                      Pro League, Championship Round                        0  
                      Pro League, Europa Playoffs                           0  
                      Pro League, Qualifying Round                          0  
                      Pro League, Relegation Round                          0  
                      Pro League, Relegation/Promotion Playoffs             0  
eliteserien    2024   Eliteserien                                           6  
                      Eliteserien, Relegation/Promotion                     1  
               2025   Eliteserien                                           4  
                      Eliteserien, Relegation/Promotion                     0  
liga_portugal  24/25  Liga Portugal                                         3  
                      Liga Portugal, Relegation/Promotion Playoffs          0  
               25/26  Liga Portugal                                         3  
                      Liga Portugal, Relegation/Promotion Playoffs          0  
premier_league 24/25  Premier League                                        6  
               25/26  Premier League                                       15

In [ ]:
matches[matches["status"] != "closed"][["league", "season", "match_id", "start_time", "status", "group_clean"]]

Findings

The regular seasons are complete.

League	        Regular-season matches (both seasons)	Expected
Premier League	            380	                           380
Liga Portugal	            306	                           306
Eliteserien	                240	                           240
Belgium	                    240	                           240

The postponed rows are leftovers. Every one of the postponed matches was replayed under a new ID, which is why the closed counts come out exactly right. Drop them. 2,551 rows, 0 duplicate IDs.

Shots and shots on target are 100% present. Possession ranges from 84% to 99.7%:

Eliteserien 2024 is the weak spot at 84.1%, which means about 39 matches are missing possession.
Everything else is at 96% or above.
Decision: keep every match in silver and leave possession as null where it’s missing. In the regression, drop only the matches with no possession, or fit one model with possession and one without, and check the shot effects don’t change. The second option is a nice robustness check to mention in the article.